# BeeGuard — Train YOLO on Colab GPU

**Before running:** `Runtime -> Change runtime type -> T4 GPU`

Workflow: zip your annotated dataset locally -> upload to Drive -> run this notebook -> download `best.pt` -> put it back on your local machine for `app.py`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install ultralytics

In [ ]:
!pip install -q ultralytics

## 3. Unzip your dataset

On your local machine first:
- Finish annotating, run `split_dataset.py` locally to get the `images/train`, `images/val`, `labels/train`, `labels/val` layout.
- Zip that whole `yolo_dataset` folder into `yolo_dataset.zip`.
- Upload `yolo_dataset.zip` to your Google Drive, e.g. into `MyDrive/BeeGuard/`.

Then set the path below to match where you uploaded it.

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/BeeGuard/yolo_dataset.zip"

!mkdir -p /content/yolo_dataset
!unzip -q "$DRIVE_ZIP_PATH" -d /content/yolo_dataset
!find /content/yolo_dataset -maxdepth 3 -type d

If the `find` output above shows an extra nested folder (e.g. `/content/yolo_dataset/yolo_dataset/images/...`), adjust `DATASET_ROOT` in the next cell to match — zips sometimes wrap contents in one extra folder.

## 4. Write data.yaml

In [ ]:
DATASET_ROOT = "/content/yolo_dataset"  # adjust if nested, see note above

data_yaml = f"""
path: {DATASET_ROOT}
train: images/train
val: images/val

names:
  0: BeeHiveFrame
  1: CappedHoneyCell
  2: CappedWorkerBroodCell
  3: PollenCell
  4: UncappedNectarCell
  5: UncappedWorkerLarvaCell
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

print(data_yaml)

## 5. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="/content/data.yaml",
    epochs=150,
    imgsz=960,
    batch=16,
    patience=30,
    device=0,
    project="/content/beeguard_model",
    name="train",
    mosaic=1.0,
    scale=0.5,
    degrees=10.0,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
)

## 6. Check per-class results

In [ ]:
best_model = YOLO("/content/beeguard_model/train/weights/best.pt")
metrics = best_model.val(data="/content/data.yaml", imgsz=960, device=0)

print("\nPer-class mAP50:")
for cls_id, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"  {best_model.names[cls_id]:28s} mAP50={ap:.3f}")

## 7. Save best.pt back to Drive (survives when the Colab runtime disconnects)

In [ ]:
!mkdir -p /content/drive/MyDrive/BeeGuard/trained_models
!cp /content/beeguard_model/train/weights/best.pt /content/drive/MyDrive/BeeGuard/trained_models/best.pt
print("Saved to Drive: MyDrive/BeeGuard/trained_models/best.pt")

## 8. Download best.pt directly to your machine (alternative to Drive)

In [ ]:
from google.colab import files
files.download("/content/beeguard_model/train/weights/best.pt")

## Next step

Move `best.pt` into `D:\\BeeGuard_Videos\\model_files\\` (or wherever), then in `app.py` locally set:

```python
CUSTOM_MODEL_PATH = r"D:\\BeeGuard_Videos\\model_files\\best.pt"
```

and run `streamlit run app.py` in your VS Code terminal, on your own machine (no GPU needed for single-image inference).